# Week 6 Assignment: Prediction Shootout
# 第 6 週作業：空間預測對決

| 事件 | 說明 | 日期 | 降雨特性 |
|------|------|------|----------|
| **事件 1** | 楊柳颱風 (Typhoon Yagi) | 2025-08-12 | 集中型颱風降雨，測站稀疏（34站） |
| **事件 2** | 0728 豪雨事件 | 2025-07-28 | 廣域均勻型強降雨，測站密集（154站） |

分析範圍：花蓮縣 + 宜蘭縣 ｜ CRS：EPSG:3826（TWD97 / TM2 Zone 121）

---
## AI 使用日誌
- 資料解析、EPSG:3826 座標轉換、四種內插方法架構由 Claude AI (claude-sonnet-4-6) 輔助撰寫
- Variogram 模型比較邏輯、參數解讀、不確定性分析文字由學生自行完成
- 所有圖表輸出與 GeoTIFF 驗證由學生確認後執行


## A0. 套件載入與路徑設定

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pyproj import Transformer
from scipy.interpolate import NearestNDInterpolator
from scipy.spatial.distance import cdist
from pykrige.ok import OrdinaryKriging
from sklearn.ensemble import RandomForestRegressor
import rasterio
from rasterio.transform import from_bounds

plt.rcParams['font.family'] = ['Microsoft JhengHei', 'DejaVu Sans', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# 路徑設定
# ============================================================
PATH_E1 = r'C:\Users\user\CascadeProjects\hw6\data\rain_20250812\rain_20250812.csv'
PATH_E2 = r'C:\Users\user\CascadeProjects\hw6\data\rain_20250728\rain_20250728.csv'
OUT_DIR = r'C:\Users\user\CascadeProjects\hw6\output'

os.makedirs(OUT_DIR, exist_ok=True)  # 若資料夾不存在則自動建立

def out(filename):
    """回傳 OUTPUT 資料夾的完整路徑"""
    return os.path.join(OUT_DIR, filename)

EVENT1_NAME     = '楊柳颱風 (2025-08-12)'
EVENT2_NAME     = '0728 豪雨事件 (2025-07-28)'
transformer     = Transformer.from_crs('EPSG:4326', 'EPSG:3826', always_xy=True)
TARGET_COUNTIES = ['花蓮縣', '宜蘭縣']

print('✅ 套件載入完成')
print(f'📁 輸出路徑：{OUT_DIR}')

In [ ]:
def load_and_prepare(path, event_name, rain_col='Past24hr'):
    """
    載入 CSV → 篩選花蓮+宜蘭 → 過濾 -998/0
    → 取每站 Past24hr 最大值快照（代表事件峰值）
    → 轉換至 EPSG:3826 公尺座標
    """
    df = pd.read_csv(path)
    df = df[df['CountyName'].isin(TARGET_COUNTIES)].copy()
    df = df[(df[rain_col] > 0) & (df[rain_col] != -998)]
    df = df.sort_values(rain_col, ascending=False).groupby('StationId').first().reset_index()
    e, n = transformer.transform(df['StationLongitude'].values, df['StationLatitude'].values)
    df['easting']  = e
    df['northing'] = n
    df['rainfall'] = df[rain_col].values
    print(f'✅ {event_name}')
    print(f'   有效測站：{len(df)} 站')
    print(f'   降雨範圍：{df["rainfall"].min():.1f} ~ {df["rainfall"].max():.1f} mm')
    print(f'   平均降雨：{df["rainfall"].mean():.1f} mm\n')
    return df

df_e1 = load_and_prepare(PATH_E1, EVENT1_NAME)
df_e2 = load_and_prepare(PATH_E2, EVENT2_NAME)

In [ ]:
def make_grid(df, res=1000, buffer=5000):
    """建立 1000m 解析度插值網格（EPSG:3826 公尺）"""
    xi = np.arange(df['easting'].min()  - buffer, df['easting'].max()  + buffer, res)
    yi = np.arange(df['northing'].min() - buffer, df['northing'].max() + buffer, res)
    XX, YY = np.meshgrid(xi, yi)
    return xi, yi, XX, YY

xi1, yi1, XX1, YY1 = make_grid(df_e1)
xi2, yi2, XX2, YY2 = make_grid(df_e2)
print(f'事件1 網格大小：{XX1.shape}')
print(f'事件2 網格大小：{XX2.shape}')

---
## A1. Variogram 分析
### A1-1 & A1-2：兩事件各比較 Spherical vs Exponential

In [ ]:
def compute_experimental_variogram(x, y, z, n_lags=15, max_dist_ratio=0.6):
    coords   = np.column_stack([x, y])
    dists    = cdist(coords, coords)
    max_dist = np.max(dists) * max_dist_ratio
    lags     = np.linspace(0, max_dist, n_lags + 1)
    gamma, centers = [], []
    for i in range(len(lags) - 1):
        mask = (dists > lags[i]) & (dists <= lags[i+1])
        if mask.sum() > 0:
            idx = np.where(mask)
            gamma.append(0.5 * np.mean((z[idx[0]] - z[idx[1]])**2))
            centers.append((lags[i] + lags[i+1]) / 2)
    return np.array(centers), np.array(gamma)


params_e1, params_e2 = {}, {}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('A1. Variogram 比較圖（Spherical vs Exponential）', fontsize=14, fontweight='bold')

for df, ax, name, pd_ in [
    (df_e1, axes[0], EVENT1_NAME, params_e1),
    (df_e2, axes[1], EVENT2_NAME, params_e2)
]:
    x = df['easting'].values  / 1000
    y = df['northing'].values / 1000
    z = df['rainfall'].values
    centers, gamma = compute_experimental_variogram(x, y, z)

    ok_s = OrdinaryKriging(x, y, z, variogram_model='spherical',   nlags=12, weight=True, verbose=False, enable_plotting=False)
    ok_e = OrdinaryKriging(x, y, z, variogram_model='exponential', nlags=12, weight=True, verbose=False, enable_plotting=False)

    def get_p(ok):
        p = ok.variogram_model_parameters  # [psill, range, nugget]
        return p[2], p[0] + p[2], p[1]     # → nugget, sill, range

    ns, ss, rs = get_p(ok_s)
    ne, se, re = get_p(ok_e)
    r_range = np.linspace(0, centers.max(), 200)
    sph_fn  = lambda h, n, s, r: np.where(h <= r, n + (s-n)*(1.5*(h/r) - 0.5*(h/r)**3), s)
    exp_fn  = lambda h, n, s, r: n + (s-n)*(1 - np.exp(-h / (r/3)))
    ssr_s   = np.sum((gamma - sph_fn(centers, ns, ss, rs))**2)
    ssr_e   = np.sum((gamma - exp_fn(centers, ne, se, re))**2)
    best    = 'spherical' if ssr_s < ssr_e else 'exponential'

    ax.scatter(centers, gamma, color='black', zorder=5, s=40, label='Experimental')
    ax.plot(r_range, sph_fn(r_range, ns, ss, rs), 'b-',  lw=2, label=f'Spherical   (SSR={ssr_s:.1f})')
    ax.plot(r_range, exp_fn(r_range, ne, se, re), 'r--', lw=2, label=f'Exponential (SSR={ssr_e:.1f})')
    ax.set_xlabel('Lag Distance (km)')
    ax.set_ylabel('Semivariance')
    ax.set_title(f'{name}\nBest fit: {best.capitalize()}', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    pd_.update({
        'best':        best,
        'spherical':   {'sill': ss, 'range': rs, 'nugget': ns, 'ssr': ssr_s},
        'exponential': {'sill': se, 'range': re, 'nugget': ne, 'ssr': ssr_e}
    })

plt.tight_layout()
plt.savefig(out('variogram_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ 已儲存：', out('variogram_comparison.png'))

### A1-3. 最佳模型選擇理由（各 1-2 句）

In [ ]:
print('=== A1-3. 最佳模型選擇理由 ===')
print()
for name, pd_ in [(EVENT1_NAME, params_e1), (EVENT2_NAME, params_e2)]:
    best = pd_['best']
    alt  = 'exponential' if best == 'spherical' else 'spherical'
    print(f'【{name}】')
    print(f'  選擇 {best.capitalize()} 模型（SSR={pd_[best]["ssr"]:.2f}），'
          f'優於 {alt.capitalize()}（SSR={pd_[alt]["ssr"]:.2f}）。')
    if best == 'exponential':
        print('  Exponential 模型以漸近方式趨近 Sill，符合降雨場「近距差異大、遠距趨穩」的'
              '空間衰減特性，實驗 Variogram 點的擬合殘差較 Spherical 更小。')
    else:
        print('  Spherical 模型在有限 Range 內完全達到 Sill，'
              '反映此降雨事件具有明確的空間相關截止距離，擬合更貼近實驗點。')
    print()

### A1-4. 兩事件 Variogram 參數差異（Sill / Range / Nugget）

In [ ]:
b1 = params_e1['best']; b2 = params_e2['best']
p1 = params_e1[b1];     p2 = params_e2[b2]

print('=== A1-4. 跨事件 Variogram 參數比較 ===')
print(f'{"參數":<12}{EVENT1_NAME:<28}{EVENT2_NAME:<28}')
print('-' * 68)
print(f'{"Sill":<12}{p1["sill"]:<28.2f}{p2["sill"]:<28.2f}')
print(f'{"Range(km)":<12}{p1["range"]:<28.2f}{p2["range"]:<28.2f}')
print(f'{"Nugget":<12}{p1["nugget"]:<28.2f}{p2["nugget"]:<28.2f}')
print(f'{"Best Model":<12}{b1.capitalize():<28}{b2.capitalize():<28}')
print()
print('📝 參數差異解讀（口訣：Sill 看天氣、Nugget 看儀器、Range 看地理）：')
print()
print('【Sill 看天氣】')
print(f'  0728 豪雨 Sill={p2["sill"]:.0f} 遠大於楊柳颱風 Sill={p1["sill"]:.0f}，')
print('  反映豪雨事件降雨強度空間差異極大（最大 171 mm vs 颱風 29 mm），')
print('  整體降雨場的總體變異遠高於颱風事件。')
print()
print('【Range 看地理】')
print(f'  楊柳颱風 Range={p1["range"]:.0f} km 遠大於 0728 豪雨 Range={p2["range"]:.0f} km。')
print('  颱風環流尺度大，空間相關延伸遠；0728 豪雨屬對流系統，局部叢聚型，空間相關截止距離短。')
print()
print('【Nugget 看儀器】')
print(f'  楊柳颱風 Nugget={p1["nugget"]:.0f} 偏高，主因是只有 34 站，')
print('  樣本稀疏造成短距離估計誤差，而非儀器本身問題。')
print(f'  0728 豪雨 Nugget≈{p2["nugget"]:.0f}，154 站密集使空間連續性近乎完美。')

---
## A2. 四種方法內插

In [ ]:
def idw_interpolation(x, y, z, XX, YY, power=2):
    """IDW 手動實作（scipy.spatial.distance.cdist），power=2"""
    grid_pts = np.column_stack([XX.ravel(), YY.ravel()])
    obs_pts  = np.column_stack([x, y])
    dists    = cdist(grid_pts, obs_pts)
    dists    = np.where(dists == 0, 1e-10, dists)
    weights  = 1.0 / dists**power
    return ((weights * z).sum(axis=1) / weights.sum(axis=1)).reshape(XX.shape)


results = {}
for label, df, xi, yi, XX, YY, pd_ in [
    ('e1', df_e1, xi1, yi1, XX1, YY1, params_e1),
    ('e2', df_e2, xi2, yi2, XX2, YY2, params_e2)
]:
    x    = df['easting'].values  / 1000
    y    = df['northing'].values / 1000
    z    = df['rainfall'].values
    xi_k = xi / 1000;  yi_k = yi / 1000
    XX_k = XX / 1000;  YY_k = YY / 1000

    print(f'🔄 執行 {label} 插值（最佳 Variogram: {pd_["best"]}）...')

    # 1. Nearest Neighbor
    nn   = NearestNDInterpolator(list(zip(x, y)), z)
    z_nn = nn(XX_k, YY_k)

    # 2. IDW (power=2)
    z_idw = idw_interpolation(x, y, z, XX_k, YY_k)

    # 3. Ordinary Kriging
    ok = OrdinaryKriging(
        x, y, z, variogram_model=pd_['best'],
        nlags=12, weight=True, verbose=False, enable_plotting=False
    )
    z_k, z_v = ok.execute('grid', xi_k, yi_k)
    z_k = np.clip(np.array(z_k), 0, None)
    z_v = np.clip(np.array(z_v), 0, None)

    # 4. Random Forest (n_estimators=200, min_samples_leaf=3)
    rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42, n_jobs=-1)
    rf.fit(np.column_stack([x, y]), z)
    z_rf = rf.predict(np.column_stack([XX_k.ravel(), YY_k.ravel()])).reshape(XX.shape)

    results[label] = {'nn': z_nn, 'idw': z_idw, 'kriging': z_k, 'kriging_var': z_v, 'rf': z_rf}
    print(f'  ✅ {label} 完成\n')

print('✅ 全部插值完成')

In [ ]:
# 2×2 四圖並列比較圖
for label, df, XX, YY, name, fname in [
    ('e1', df_e1, XX1, YY1, EVENT1_NAME, 'event1_interpolation_comparison.png'),
    ('e2', df_e2, XX2, YY2, EVENT2_NAME, 'event2_interpolation_comparison.png')
]:
    res  = results[label]
    vmax = max(res['nn'].max(), res['idw'].max(), res['kriging'].max(), res['rf'].max())
    fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    fig.suptitle(f'A2. 四種內插方法比較\n{name}', fontsize=13, fontweight='bold')
    for ax, (key, title) in zip(axes.flat, [
        ('nn',      'Nearest Neighbor'),
        ('idw',     'IDW (power=2)'),
        ('kriging', 'Ordinary Kriging'),
        ('rf',      'Random Forest (n=200)')
    ]):
        im = ax.pcolormesh(XX/1000, YY/1000, res[key], cmap='YlOrRd', vmin=0, vmax=vmax, shading='auto')
        ax.scatter(df['easting']/1000, df['northing']/1000,
                   c=df['rainfall'], cmap='YlOrRd', vmin=0, vmax=vmax,
                   edgecolors='black', linewidths=0.5, s=40, zorder=5)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xlabel('Easting (km)'); ax.set_ylabel('Northing (km)')
        plt.colorbar(im, ax=ax, label='Rainfall (mm)', fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(out(fname), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ 已儲存：{out(fname)}')

In [ ]:
# Kriging vs RF 差異圖（RdBu_r colormap）
for label, XX, YY, name, fname in [
    ('e1', XX1, YY1, EVENT1_NAME, 'event1_kriging_vs_rf_diff.png'),
    ('e2', XX2, YY2, EVENT2_NAME, 'event2_kriging_vs_rf_diff.png')
]:
    diff    = results[label]['kriging'] - results[label]['rf']
    abs_max = np.percentile(np.abs(diff), 95)
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.pcolormesh(XX/1000, YY/1000, diff, cmap='RdBu_r', vmin=-abs_max, vmax=abs_max, shading='auto')
    ax.set_title(f'Kriging − RF 差異圖\n{name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Easting (km)'); ax.set_ylabel('Northing (km)')
    plt.colorbar(im, ax=ax, label='Kriging - RF (mm)')
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(out(fname), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ 已儲存：{out(fname)}')

---
## A3. 不確定性分析 — Sigma Map

In [ ]:
sigma_e1 = np.sqrt(results['e1']['kriging_var'])
sigma_e2 = np.sqrt(results['e2']['kriging_var'])

for sigma, XX, YY, name, fname in [
    (sigma_e1, XX1, YY1, EVENT1_NAME, 'event1_sigma_map.png'),
    (sigma_e2, XX2, YY2, EVENT2_NAME, 'event2_sigma_map.png')
]:
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.pcolormesh(XX/1000, YY/1000, sigma, cmap='Blues', shading='auto')
    ax.set_title(f'A3. Sigma Map（Kriging 預測標準差）\n{name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Easting (km)'); ax.set_ylabel('Northing (km)')
    plt.colorbar(im, ax=ax, label='Kriging Std. Dev. (mm)')
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(out(fname), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ 已儲存：{out(fname)}')
    print(f'   Sigma 均值={sigma.mean():.2f} mm，最大={sigma.max():.2f} mm\n')

### A3. 比較分析（300 字以內）

In [ ]:
print('=== A3. 不確定性比較分析 ===')
print(f'事件1 Sigma：均值={sigma_e1.mean():.2f} mm，最大={sigma_e1.max():.2f} mm')
print(f'事件2 Sigma：均值={sigma_e2.mean():.2f} mm，最大={sigma_e2.max():.2f} mm')
print()
print("""
【Q1 兩事件 Sigma Map 差異？為什麼？】
楊柳颱風（事件1）僅 34 站，測站稀疏使多數網格點距最近觀測站較遠，
山區與南部邊界出現大面積高 Sigma。
0728 豪雨（事件2）有 154 站、分布密集，大多數網格鄰近多個觀測值，
Sigma 整體偏低，僅網格邊緣略有升高。

【Q2 哪種降雨 Kriging 信心較高？】
廣域均勻型豪雨（0728）測站多、覆蓋密，Kriging 預測信心顯著較高；
集中型颱風降雨（楊柳）因測站稀疏、極端值影響，不確定性明顯較大。

【Q3 指揮官在高 variance 區的決策？】
高 Sigma 區域（偏遠山谷、測站空白帶）應採保守策略：
預測雨量上調一個標準差、提前啟動疏散預警，
並即時整合雷達回波與上游水位資料補強情資。

【Q4 Random Forest 能否提供類似的不確定性資訊？】
RF 標準輸出不含空間預測不確定性。雖可用「樹間預測標準差」作代理，
但 RF 在訓練範圍外的外插區會系統低估不確定性（模型回傳訓練葉節點均值，
對無觀測區缺乏「我不確定」的訊號）。
Kriging variance 有地理統計理論保證，在稀疏測站情境下更誠實反映預測風險，
是防災決策中更可靠的不確定性量化工具。
""")

---
## A4. GeoTIFF 輸出（事件2：0728 豪雨）

In [ ]:
def export_geotiff(array, xi, yi, filename, epsg=3826):
    """
    匯出 GeoTIFF（EPSG:3826）
    重要：numpy row 0 = south，GeoTIFF row 0 = north → 需 np.flipud()
    """
    arr       = np.flipud(array).astype(np.float32)  # ← y 軸翻轉
    res       = xi[1] - xi[0]
    transform = from_bounds(
        xi.min(), yi.min(),
        xi.max() + res, yi.max() + res,
        arr.shape[1], arr.shape[0]
    )
    with rasterio.open(
        filename, 'w', driver='GTiff',
        height=arr.shape[0], width=arr.shape[1],
        count=1, dtype='float32',
        crs=f'EPSG:{epsg}', transform=transform, nodata=-9999
    ) as dst:
        dst.write(arr, 1)
    print(f'✅ 已儲存：{filename}  |  shape={arr.shape}  |  CRS=EPSG:{epsg}')


export_geotiff(results['e2']['kriging'],     xi2, yi2, out('kriging_rainfall.tif'))
export_geotiff(results['e2']['kriging_var'], xi2, yi2, out('kriging_variance.tif'))
export_geotiff(results['e2']['rf'],          xi2, yi2, out('rf_rainfall.tif'))

---
## A5. (加分) 跨事件 Variogram 綜合比較

In [ ]:
p1 = params_e1[params_e1['best']]
p2 = params_e2[params_e2['best']]

summary = pd.DataFrame({
    '參數':              ['Sill', 'Range (km)', 'Nugget', 'Best Model'],
    '事件1 楊柳颱風':    [
        f"{p1['sill']:.2f}", f"{p1['range']:.2f}",
        f"{p1['nugget']:.2f}", params_e1['best'].capitalize()
    ],
    '事件2 0728 豪雨':   [
        f"{p2['sill']:.2f}", f"{p2['range']:.2f}",
        f"{p2['nugget']:.2f}", params_e2['best'].capitalize()
    ],
    '差異原因': [
        '豪雨梯度極大（最大171mm vs 颱風29mm），整體變異遠高於颱風',
        '颱風環流尺度大→Range長；對流豪雨局部叢聚→Range短',
        '豪雨154站密集→Nugget≈0；颱風34站稀疏→短距估計誤差大→Nugget偏高',
        '兩事件均以Exponential最佳，符合漸近式空間衰減'
    ]
})

print('=== A5. 跨事件 Variogram 參數比較表 ===')
print(summary.to_string(index=False))

In [ ]:
print('=== A5. 加分問答 ===')
print()
print('Q：如果只能用一組 Variogram 參數套用到未來所有事件，你會怎麼選？為什麼有風險？')
print()
print("""
A：若強制使用單一參數，建議「保守中間值」策略：
   - Range  → 取兩事件平均（≈78 km），避免單一極端值主導
   - Sill   → 取較大值（以豪雨事件為準），保守高估不確定性
   - Nugget → 取較大值（以颱風為準），涵蓋最壞測站稀疏情境
   - Model  → Exponential（兩事件皆最佳，適用性較廣）

然而這樣做有三大風險：

1. 天氣型態多樣性
   颱風 / 梅雨 / 對流系統空間結構根本不同。
   固定短 Range 會使颱風分析過度平滑；固定長 Range 則稀釋對流豪雨熱點。

2. Sill 錯估 → 不確定性失真
   以低 Sill 套用強降雨，會系統低估遠距插值的 Kriging variance，
   讓決策者在高風險區誤判預測可信，可能延誤疏散時機。

3. 模型類型錯置
   強對流事件可能更接近 Gaussian 模型；
   強制 Exponential 會有系統偏差，影響插值品質與 Sigma 估計。

結論：Variogram 應依事件類型動態估計；固定參數僅適用快速初估，
不宜作為正式防災決策依據。
""")

In [ ]:
# ============================================================
# 繳交清單最終確認
# ============================================================
print(f'=== 繳交清單確認 ===')
print(f'輸出資料夾：{OUT_DIR}\n')
files = [
    'variogram_comparison.png',
    'event1_interpolation_comparison.png',
    'event2_interpolation_comparison.png',
    'event1_kriging_vs_rf_diff.png',
    'event2_kriging_vs_rf_diff.png',
    'event1_sigma_map.png',
    'event2_sigma_map.png',
    'kriging_rainfall.tif',
    'kriging_variance.tif',
    'rf_rainfall.tif',
]
for f in files:
    path   = out(f)
    exists = os.path.exists(path)
    size   = f'({os.path.getsize(path)/1024:.0f} KB)' if exists else ''
    print(f'  {"✅" if exists else "❌"}  {f} {size}')
print()
print('🎉 Part A 全部完成！')